# Finetuning, Instruction Tuning, and RLHF

This notebook explores the **post-pretraining pipeline** that transforms a raw language model into a helpful, harmless, and honest assistant. We cover three key stages:

1. **Supervised Finetuning (SFT)** -- Train on curated (prompt, response) pairs
2. **Instruction Tuning** -- Generalize across diverse task formats
3. **Reinforcement Learning from Human Feedback (RLHF)** -- Align the model with human preferences using a learned reward model

```
Pretrained LM  -->  SFT  -->  Reward Model  -->  RLHF (PPO)  -->  Aligned Model
     |                |              |                 |                 |
  raw text      task demos     human prefs       policy opt.      helpful output
```

Each stage introduces different tradeoffs in convergence speed, stability, and final quality.

## 1. Setup and Imports

In [ ]:
%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

from finetuning import (
    RewardModel,
    FinetuningSampler,
    analyze_instruction_tuning_formats,
    analyze_rlhf_dynamics,
)

print("All imports successful.")

---
## 2. Supervised Finetuning (SFT)

SFT is the simplest post-pretraining step. We take a pretrained model and train it on high-quality (instruction, response) pairs using the standard cross-entropy loss.

**Key characteristics:**
- Fast convergence (exponential decay)
- Low variance in the loss signal
- Task-specific -- the model becomes very good at the training distribution
- Risk of overfitting if the dataset is small

In [ ]:
# Create a sampler and run SFT simulation step by step
sampler = FinetuningSampler()

sft_losses = []
for step in range(200):
    loss = sampler.supervised_finetuning_step(step)
    sft_losses.append(loss)

print(f"SFT loss at step   0: {sft_losses[0]:.4f}")
print(f"SFT loss at step  50: {sft_losses[50]:.4f}")
print(f"SFT loss at step 100: {sft_losses[100]:.4f}")
print(f"SFT loss at step 199: {sft_losses[-1]:.4f}")

In [ ]:
# Visualize the SFT loss trajectory
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sft_losses, linewidth=2, color='steelblue', label='SFT Loss')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Asymptotic floor (0.5)')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Supervised Finetuning: Loss Trajectory')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSFT converges quickly: 90% of improvement happens in the first ~100 steps.")
print(f"The loss formula is: L(t) = 2.0 * exp(-t/50) + 0.5")

**Observation:** SFT loss decays exponentially and reaches a stable floor. This is the hallmark of supervised training on a fixed dataset -- deterministic, smooth, and predictable. The floor at 0.5 represents irreducible noise in the training signal.

---
## 3. Instruction Tuning

Instruction tuning extends SFT by training on **many different task formats** simultaneously. Instead of a single task, the model sees:

- Question-answering pairs
- Task descriptions with inputs and outputs
- Chain-of-thought reasoning examples
- Multi-task instructions

This makes convergence **slightly slower** but leads to **better generalization** across unseen tasks.

In [ ]:
# Show different instruction tuning formats
print("Common Instruction Tuning Formats")
print("=" * 60)

formats = {
    "QA Format": (
        "Q: What is the capital of France?\n"
        "A: The capital of France is Paris."
    ),
    "Task+Input Format": (
        "Task: Translate the following sentence to French.\n"
        "Input: Hello, how are you?\n"
        "Output: Bonjour, comment allez-vous?"
    ),
    "Chain-of-Thought (COT)": (
        "Question: If a store has 15 apples and sells 7, how many remain?\n"
        "Let's think step by step.\n"
        "1. The store starts with 15 apples.\n"
        "2. It sells 7 apples.\n"
        "3. 15 - 7 = 8 apples remain.\n"
        "Answer: 8"
    ),
    "Multi-task Format": (
        "Task: Sentiment analysis\n"
        "Instance: This movie was absolutely wonderful!\n"
        "Output: Positive"
    ),
}

for name, example in formats.items():
    print(f"\n--- {name} ---")
    print(example)

In [ ]:
# Simulate instruction tuning and compare with SFT
sampler2 = FinetuningSampler()
it_losses = []
sft_losses2 = []

for step in range(200):
    sft_losses2.append(sampler2.supervised_finetuning_step(step))
    it_losses.append(sampler2.instruction_tuning_step(step))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sft_losses2, linewidth=2, label='SFT (single task)', alpha=0.8)
ax.plot(it_losses, linewidth=2, label='Instruction Tuning (multi-task)', alpha=0.8)
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('SFT vs. Instruction Tuning: Convergence Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"SFT final loss:               {sft_losses2[-1]:.4f}")
print(f"Instruction tuning final loss: {it_losses[-1]:.4f}")
print(f"\nInstruction tuning converges more slowly (tau=80 vs tau=50)")
print(f"but reaches a lower asymptotic loss (0.3 vs 0.5).")

### Instruction Format Effectiveness

Not all instruction formats are equally effective. Chain-of-thought formatting typically leads to the best performance because it teaches the model to reason step by step.

In [ ]:
# Analyze instruction tuning formats using the module function
format_scores = analyze_instruction_tuning_formats()

In [ ]:
# Print a summary of format scores (lower = better in this simulation)
print("Format Effectiveness Ranking (lower score = better):")
print("-" * 45)
for rank, (fmt, score) in enumerate(sorted(format_scores.items(), key=lambda x: x[1]), 1):
    print(f"  {rank}. {fmt:<22s} score = {score:.2f}")

print("\nKey takeaway: Structured formats (COT, Task+Input) outperform")
print("unstructured text. Chain-of-thought is the most effective format.")

---
## 4. Reward Model

Before we can do RLHF, we need a **reward model** -- a neural network trained on human preference data to predict how good a response is.

The reward model architecture:
```
Hidden states (batch, seq_len, 128)
    --> Average Pool --> (batch, 128)
    --> Linear(128, 256) --> ReLU
    --> Linear(256, 128) --> ReLU
    --> Linear(128, 1)
    --> Scalar reward
```

The model is trained on pairs of responses where humans indicate which is better.

In [ ]:
# Create a reward model and inspect its architecture
torch.manual_seed(42)
reward_model = RewardModel(hidden_dim=128)
print(reward_model)
print(f"\nTotal parameters: {sum(p.numel() for p in reward_model.parameters()):,}")

In [ ]:
# Feed random hidden states through the reward model
torch.manual_seed(42)
batch_size = 4
seq_len = 16
hidden_dim = 128

hidden_states = torch.randn(batch_size, seq_len, hidden_dim)
print(f"Input shape:  {hidden_states.shape}  (batch, seq_len, hidden_dim)")

with torch.no_grad():
    rewards = reward_model(hidden_states)

print(f"Output shape: {rewards.shape}  (batch, 1) -- one scalar per sequence")
print(f"\nReward scores for each item in the batch:")
for i, r in enumerate(rewards):
    print(f"  Sample {i}: reward = {r.item():.4f}")

### How the Reward Model Scores Responses

In practice, the reward model is used to compare two candidate responses to the same prompt. The response with the higher reward is considered "better" according to the learned human preferences.

In [ ]:
# Simulate scoring different quality responses
torch.manual_seed(42)

# Generate synthetic hidden states representing responses of varying quality
# Higher magnitude hidden states simulate more confident/better responses
poor_response = torch.randn(1, 10, 128) * 0.1     # Low magnitude -- uncertain
avg_response = torch.randn(1, 10, 128) * 0.5      # Medium magnitude
good_response = torch.randn(1, 10, 128) * 1.0     # Standard magnitude
great_response = torch.randn(1, 10, 128) * 2.0    # High magnitude -- confident

responses = {
    'Poor (low magnitude)': poor_response,
    'Average (med magnitude)': avg_response,
    'Good (std magnitude)': good_response,
    'Great (high magnitude)': great_response,
}

print("Reward Model Scoring of Different Responses")
print("=" * 50)

scores = {}
with torch.no_grad():
    for name, hidden in responses.items():
        score = reward_model(hidden).item()
        scores[name] = score
        print(f"  {name:<30s} -> reward = {score:+.4f}")

print("\nNote: The untrained reward model assigns arbitrary scores.")
print("After training on human preferences, higher rewards would")
print("correlate with responses humans prefer.")

In [ ]:
# Visualize reward scores
fig, ax = plt.subplots(figsize=(10, 5))
names = list(scores.keys())
vals = list(scores.values())
colors = ['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c']

bars = ax.bar(names, vals, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Reward Score')
ax.set_title('Reward Model: Scores for Different Response Qualities')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val:+.3f}', ha='center', va='bottom' if val >= 0 else 'top',
            fontweight='bold', fontsize=10)

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## 5. RLHF Training Simulation

RLHF uses the reward model to provide a training signal to the language model via **Proximal Policy Optimization (PPO)**. The training loop:

1. Generate response from the current policy (language model)
2. Score it with the reward model
3. Update the policy to increase reward while staying close to the original model (KL penalty)

**Key characteristics:**
- More volatile loss (RL is inherently noisy)
- Slower convergence than SFT
- Can achieve better final alignment with human preferences
- Risk of reward hacking if not carefully constrained

In [ ]:
# Simulate RLHF training step by step
sampler_rlhf = FinetuningSampler()
rlhf_losses = []

for step in range(200):
    loss = sampler_rlhf.rlhf_step(step)
    rlhf_losses.append(loss)

print(f"RLHF loss at step   0: {rlhf_losses[0]:.4f}")
print(f"RLHF loss at step  50: {rlhf_losses[50]:.4f}")
print(f"RLHF loss at step 100: {rlhf_losses[100]:.4f}")
print(f"RLHF loss at step 199: {rlhf_losses[-1]:.4f}")
print(f"\nNotice the oscillations due to the sinusoidal noise term:")
print(f"  noise(t) = sin(t/5) * 0.2")

In [ ]:
# Visualize the RLHF loss trajectory
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw RLHF loss
axes[0].plot(rlhf_losses, linewidth=1.5, color='green', alpha=0.7, label='RLHF Loss (raw)')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('RLHF Training: Raw Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Smoothed RLHF loss
window = 15
smoothed = np.convolve(rlhf_losses, np.ones(window)/window, mode='valid')
axes[1].plot(smoothed, linewidth=2, color='darkgreen', label=f'RLHF Loss (smoothed, w={window})')
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Smoothed Loss')
axes[1].set_title('RLHF Training: Smoothed Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The raw RLHF loss is noisy, but the smoothed version shows")
print("a clear downward trend toward a low asymptotic value.")

### RLHF Dynamics: Reward vs. KL Divergence

A critical tension in RLHF is the tradeoff between **maximizing reward** and **staying close to the base model** (measured by KL divergence). If the policy drifts too far from the base model, it may:

- Exploit the reward model ("reward hacking")
- Lose general language capabilities
- Produce degenerate outputs that score high on the reward model but are nonsensical

In [ ]:
# Analyze RLHF dynamics: reward signal and KL divergence
analyze_rlhf_dynamics()

In [ ]:
# Discuss the reward-KL tradeoff
print("RLHF Reward-KL Tradeoff")
print("=" * 50)
print()
print("As training progresses:")
print("  - Reward increases (the model learns what humans prefer)")
print("  - KL divergence grows (the model deviates from the base policy)")
print()
print("The KL penalty in the PPO objective acts as a regularizer:")
print("  objective = E[reward] - beta * KL(policy || base_policy)")
print()
print("If KL divergence exceeds the threshold (~0.2), training")
print("should be stopped or the KL coefficient (beta) increased.")

---
## 6. Full Comparison: SFT vs. Instruction Tuning vs. RLHF

Now let us run all three finetuning approaches together and compare their training dynamics side by side.

In [ ]:
# Simulate all three finetuning methods together
full_sampler = FinetuningSampler()
history = full_sampler.simulate_finetuning(num_steps=200)

print("Simulation complete. Loss history collected for all methods.")
print(f"  Steps simulated: {len(history['supervised'])}")
print(f"  Methods: {list(history.keys())}")

In [ ]:
# Plot raw and smoothed training curves
full_sampler.plot_training_curves()

In [ ]:
# Compare final performance across methods
full_sampler.compare_final_performance()

---
## 7. Detailed Loss Analysis

In [ ]:
# Analyze convergence rates
print("Convergence Analysis")
print("=" * 60)
print()

for name, losses in history.items():
    losses_arr = np.array(losses)
    initial = losses_arr[0]
    final = losses_arr[-1]
    best = losses_arr.min()
    worst = losses_arr.max()
    variance = losses_arr[100:].var()  # Variance in second half
    
    # Find step where loss first drops below 1.0
    below_1 = np.where(losses_arr < 1.0)[0]
    first_below_1 = below_1[0] if len(below_1) > 0 else 'never'
    
    print(f"  {name.upper()}:")
    print(f"    Initial loss:       {initial:.4f}")
    print(f"    Final loss:         {final:.4f}")
    print(f"    Best loss:          {best:.4f}")
    print(f"    Improvement:        {initial - final:.4f} ({(1 - final/initial)*100:.1f}%)")
    print(f"    Variance (2nd half):{variance:.6f}")
    print(f"    First step < 1.0:   {first_below_1}")
    print()

In [ ]:
# Plot the convergence rates on a log scale
fig, ax = plt.subplots(figsize=(10, 5))

for name, losses in history.items():
    # Subtract the asymptotic floor to see the convergence rate
    ax.semilogy(losses, linewidth=2, label=name, alpha=0.7)

ax.set_xlabel('Training Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Convergence Rates (Log Scale)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Reward Model Internals

In [ ]:
# Examine the reward model layer by layer
torch.manual_seed(42)
rm = RewardModel(hidden_dim=128)

print("Reward Model Layer Inspection")
print("=" * 50)
for name, param in rm.named_parameters():
    print(f"  {name:<20s} shape={str(list(param.shape)):<16s} params={param.numel():>6,}")

total = sum(p.numel() for p in rm.parameters())
print(f"\n  {'TOTAL':<20s} {'':16s} params={total:>6,}")

In [ ]:
# Show how average pooling works in the reward model
torch.manual_seed(42)

# Simulate a single sequence with 8 tokens and hidden_dim=128
seq = torch.randn(1, 8, 128)
print(f"Input sequence shape: {seq.shape}")

# Step 1: Average pooling across the sequence dimension
pooled = seq.mean(dim=1)
print(f"After avg pooling:    {pooled.shape}")
print(f"Pooled vector (first 10 dims): {pooled[0, :10].tolist()}")

# Step 2: Pass through MLP
with torch.no_grad():
    reward = rm(seq)
print(f"\nFinal reward score:   {reward.item():.4f}")
print(f"\nThe sequence is collapsed to a single vector via average pooling,")
print(f"then mapped to a scalar reward through the 3-layer MLP.")

---
## 9. The Complete Finetuning Pipeline

In practice, modern LLM alignment follows this pipeline:

```
Stage 1: Pretraining
  - Train on massive unlabeled text (trillions of tokens)
  - Learns general language understanding
  - Objective: next-token prediction

Stage 2: Supervised Finetuning (SFT)
  - Train on curated (instruction, response) pairs
  - Learns to follow instructions
  - Objective: cross-entropy on response tokens

Stage 3: Reward Model Training
  - Train on human preference data (response A > response B)
  - Learns to score response quality
  - Objective: Bradley-Terry model (pairwise ranking loss)

Stage 4: RLHF (PPO)
  - Optimize policy to maximize reward model score
  - KL penalty prevents drift from SFT model
  - Objective: E[reward] - beta * KL(policy || reference)
```

In [ ]:
# Visualize the pipeline stages with simulated quality metrics
stages = ['Pretrained', 'After SFT', 'After IT', 'After RLHF']
helpfulness = [0.2, 0.6, 0.7, 0.9]
harmlessness = [0.3, 0.5, 0.6, 0.85]
honesty = [0.4, 0.55, 0.65, 0.8]

x = np.arange(len(stages))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, helpfulness, width, label='Helpfulness', color='steelblue', alpha=0.8)
bars2 = ax.bar(x, harmlessness, width, label='Harmlessness', color='orange', alpha=0.8)
bars3 = ax.bar(x + width, honesty, width, label='Honesty', color='green', alpha=0.8)

ax.set_xlabel('Pipeline Stage')
ax.set_ylabel('Score')
ax.set_title('Model Quality Across the Finetuning Pipeline (Simulated)')
ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Each stage adds a specific capability:")
print("  - SFT: Large jump in helpfulness (learns to follow instructions)")
print("  - IT:  Modest gains across all dimensions (generalizes)")
print("  - RLHF: Significant improvement in all dimensions (aligns with humans)")

---
## 10. Analysis: Tradeoffs in Finetuning Methods

In [ ]:
print("Tradeoff Analysis: SFT vs Instruction Tuning vs RLHF")
print("=" * 65)
print()
print(f"{'Property':<25s} {'SFT':<15s} {'Instr. Tuning':<15s} {'RLHF':<15s}")
print("-" * 65)
print(f"{'Convergence Speed':<25s} {'Fast':<15s} {'Medium':<15s} {'Slow':<15s}")
print(f"{'Training Stability':<25s} {'High':<15s} {'High':<15s} {'Low (noisy)':<15s}")
print(f"{'Final Loss':<25s} {'~0.50':<15s} {'~0.30':<15s} {'~0.20':<15s}")
print(f"{'Generalization':<25s} {'Low':<15s} {'High':<15s} {'Medium':<15s}")
print(f"{'Data Requirements':<25s} {'Task-specific':<15s} {'Multi-task':<15s} {'Preferences':<15s}")
print(f"{'Compute Cost':<25s} {'Low':<15s} {'Medium':<15s} {'High':<15s}")
print(f"{'Risk':<25s} {'Overfitting':<15s} {'Data quality':<15s} {'Reward hack':<15s}")
print()
print("Key insight: These methods are complementary, not competing.")
print("The standard pipeline uses all three in sequence.")

In [ ]:
# Visualize the stability-quality tradeoff
fig, ax = plt.subplots(figsize=(8, 6))

methods_data = {
    'SFT': {'stability': 0.95, 'quality': 0.6, 'color': 'steelblue'},
    'Instruction Tuning': {'stability': 0.85, 'quality': 0.75, 'color': 'orange'},
    'RLHF': {'stability': 0.5, 'quality': 0.92, 'color': 'green'},
}

for name, data in methods_data.items():
    ax.scatter(data['stability'], data['quality'], s=300,
               color=data['color'], label=name, edgecolors='black',
               linewidth=1.5, zorder=5)
    ax.annotate(name, (data['stability'], data['quality']),
                textcoords='offset points', xytext=(15, 10),
                fontsize=11, fontweight='bold')

ax.set_xlabel('Training Stability', fontsize=12)
ax.set_ylabel('Final Output Quality', fontsize=12)
ax.set_title('Stability vs. Quality Tradeoff', fontsize=14)
ax.set_xlim(0.3, 1.05)
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.3)

# Add an arrow showing the pipeline direction
ax.annotate('', xy=(0.5, 0.92), xytext=(0.95, 0.6),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--'))
ax.text(0.78, 0.72, 'Pipeline\ndirection', fontsize=9, color='gray',
        ha='center', style='italic')

plt.tight_layout()
plt.show()

---
## 11. Key Insights

### The Finetuning Pipeline

**Supervised Finetuning (SFT):**
- The workhorse of model adaptation. Trains on curated (prompt, response) pairs.
- Fast convergence with smooth, predictable loss curves.
- Limitation: only learns from the specific examples it sees -- limited generalization.

**Instruction Tuning:**
- Extension of SFT across diverse task formats (QA, translation, reasoning, etc.).
- Chain-of-thought formatting is the most effective instruction format.
- Slightly slower convergence but significantly better generalization.

**Reward Model:**
- A separate neural network that learns to predict human preferences.
- Takes hidden states, produces a scalar reward score.
- Trained on pairwise comparisons ("response A is better than B").

**RLHF:**
- Uses the reward model to optimize the language model via PPO.
- More volatile training (inherent to RL) but can reach the best final quality.
- Must balance reward maximization against KL divergence from the base model.
- Risk of reward hacking if the KL constraint is too loose.

### Practical Considerations

| Aspect | Recommendation |
|--------|---------------|
| When to use SFT | Single task, limited data, need fast results |
| When to use IT | Multi-task generalization, building a general assistant |
| When to use RLHF | Maximum alignment with human preferences |
| KL threshold | Monitor KL divergence; stop if it exceeds ~0.2 |
| Pipeline order | Always: Pretrain -> SFT -> RLHF (not the reverse) |

In [ ]:
print("Summary")
print("=" * 60)
print()
print("The modern LLM alignment pipeline:")
print("  1. Pretraining    -> General language understanding")
print("  2. SFT            -> Instruction following")
print("  3. Reward Model   -> Learns human preferences")
print("  4. RLHF (PPO)     -> Aligns model with preferences")
print()
print("Each stage is complementary:")
print("  - SFT provides a stable foundation (fast, smooth convergence)")
print("  - Instruction tuning adds generalization (diverse task formats)")
print("  - RLHF refines alignment (volatile but best final quality)")
print()
print("The key tradeoff: stability vs. quality.")
print("SFT is the most stable; RLHF achieves the best final results.")
print("Using all three together yields the best aligned models.")